# CPS ASEC Summary Data

Processes raw CPS ASEC microdata for income years 2014-2025 and generates
summary CSVs for use by the US Chartbook.

**Outputs:** 6 CSV files pushed to `bdecon/econ_data` on GitHub.

**Data source:** Census Bureau CPS Annual Social and Economic Supplement

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

ASEC_PATH = Path('/home/brian/Documents/ASEC/data/')
OUTPUT_PATH = Path('/home/brian/Documents/econ_data/micro/')

In [2]:
# Variables needed from ASEC person records
CORE_VARS = ['H_SEQ', 'PPPOS', 'PRECORD', 'A_AGE', 'MARSUPWT', 'PERLIS']

WORK_VARS = ['WKSWORK', 'PRDISFLG', 'RSNNOTW', 'PYRSN', 'LKWEEKS']

INCOME_VARS = ['PEARNVAL', 'ERN_VAL', 'PTOTVAL']

SPM_CORE = [
    'SPM_Resources', 'SPM_PovThreshold', 'SPM_Poor',
    'SPM_Weight', 'SPM_ID', 'SPM_NumPer', 'SPM_Totval',
    'SPM_CapWkCCXpns', 'SPM_MedXpns', 'SPM_ChildSupPd',
]

SPM_PROGRAMS = [
    'SPM_ACTC', 'SPM_EITC', 'SPM_FedTax', 'SPM_FedTaxBC',
    'SPM_FICA', 'SPM_SNAPSub', 'SPM_SchLunch', 'SPM_WICval',
    'SPM_EngVal', 'SPM_BBSUBVAL', 'SPM_CapHouseSub',
    'SPM_ChildcareXpns', 'SPM_WkXpns', 'SPM_EIP',
]

CASH_BENEFITS = ['SS_VAL', 'SSI_VAL', 'UC_VAL', 'VET_VAL',
                 'WC_VAL', 'PAW_VAL', 'CSP_VAL']

ALL_VARS = CORE_VARS + WORK_VARS + INCOME_VARS + SPM_CORE + SPM_PROGRAMS + CASH_BENEFITS

# Stata-to-SPM column mapping (for income years 2014-2017)
STATA_MAP = {
    'spmu_id': 'SPM_ID', 'spmu_poor': 'SPM_Poor',
    'spmu_povthreshold': 'SPM_PovThreshold',
    'spmu_numper': 'SPM_NumPer',
    'spmu_resources': 'SPM_Resources',
    'spmu_totval': 'SPM_Totval',
    'spmu_weight': 'SPM_Weight',
    'spmu_snapsub': 'SPM_SNAPSub',
    'spmu_caphousesub': 'SPM_CapHouseSub',
    'spmu_schlunch': 'SPM_SchLunch',
    'spmu_engval': 'SPM_EngVal',
    'spmu_wicval': 'SPM_WICval',
    'spmu_fedtax': 'SPM_FedTax',
    'spmu_fedtaxbc': 'SPM_FedTaxBC',
    'spmu_eitc': 'SPM_EITC',
    'spmu_actc': 'SPM_ACTC',
    'spmu_fica': 'SPM_FICA',
    'spmu_capwknchcarexpns': 'SPM_CapWkCCXpns',
    'spmu_wkxpns': 'SPM_WkXpns',
    'spmu_childcare': 'SPM_ChildcareXpns',
    'spmu_medoopnmcareb': 'SPM_MedXpns',
    'spmu_childsuppd': 'SPM_ChildSupPd',
}

# ASEC file years available and their dictionary types
# asec_year = income_year + 1
# Era A/B (2015-2018): dd.txt dictionaries, Stata SPM merge
# Era C (2019-2026): persfmt format, SPM built in
DICT_FILES = {
    2015: ('dd', 'asec2015early_pubuse.dd.txt'),
    2016: ('dd', 'asec2016_data_dict_full.txt'),
    2017: ('dd', '08ASEC2017_Data_Dict_Full.txt'),
    2018: ('dd', '08ASEC2018_Data_Dict_Full.txt'),
    2019: ('persfmt', 'persfmt.txt'),
    2020: ('persfmt', 'persfmt20.txt'),
    2021: ('persfmt', 'persfmt21.txt'),
    2022: ('persfmt', 'persfmt22.txt'),
    2023: ('persfmt', 'persfmt23.txt'),
    2024: ('persfmt', 'persfmt24.txt'),
    2025: ('persfmt', 'persfmt25.txt'),
    2026: ('persfmt', 'persfmt26.txt'),
}

DATA_FILES = {
    2015: 'asec2015_pubuse.dat',
    2016: 'asec2016_pubuse_v3.dat',
    2017: 'asec2017_pubuse.dat',
    2018: 'asec2018_pubuse.dat',
    2019: 'asec2019_pubuse.dat',
    2020: 'asec2020_pubuse.dat',
    2021: 'asec2021_pubuse.dat',
    2022: 'asec2022_pubuse.dat',
    2023: 'asec2023_pubuse.dat',
    2024: 'asec2024_pubuse.dat',
    2025: 'asec2025_pubuse.dat',
    2026: 'asec2026_pubuse.dat',
}

In [3]:
def read_asec_person(asec_year, path=ASEC_PATH):
    """Read ASEC person records for a given ASEC file year.
    
    Income year = asec_year - 1.
    Returns standardized DataFrame with consistent column names.
    """
    era, dictname = DICT_FILES[asec_year]
    dictfile = path / dictname
    datafile = path / DATA_FILES[asec_year]
    
    if era == 'persfmt':
        # Era C: SPM built into public use file.
        # Case-insensitive match: starting with the ASEC 2026 file, Census
        # renamed SPM_* variables to all-uppercase (e.g. SPM_Poor -> SPM_POOR).
        dd = open(dictfile).read()
        p = re.compile(f'({"|".join(ALL_VARS)})\\s+(\\d+)\\s+(\\d+)\\s',
                       re.IGNORECASE)
        name_map = {v.upper(): v for v in ALL_VARS}
        cols = {name_map[name.upper()]: (int(start) - 1, int(start) - 1 + int(length))
                for name, length, start in re.findall(p, dd)}
        
        df = (pd.read_fwf(datafile,
                          colspecs=list(cols.values()),
                          header=None,
                          names=list(cols.keys()))
                .query('PRECORD == 3')
                .apply(pd.to_numeric, errors='coerce'))
    
    else:
        # Era A/B: dd.txt format, no SPM in data file
        non_spm = [v for v in ALL_VARS if not v.startswith('SPM_')]
        dd = open(dictfile, encoding='iso-8859-1').read()
        p = re.compile(f'D ({"|".join(non_spm)})\\s+(\\d+)\\s+(\\d+)\\s+')
        cols = {name: (int(start) - 1, int(start) - 1 + int(length))
                for name, length, start in re.findall(p, dd)}
        
        df = (pd.read_fwf(datafile,
                          colspecs=list(cols.values()),
                          header=None,
                          names=list(cols.keys()))
                .query('PRECORD == 3')
                .apply(pd.to_numeric, errors='coerce'))
        
        # Merge Stata SPM data
        income_year = asec_year - 1
        spm = pd.read_stata(path / f'spmresearch{income_year}.dta')
        df = pd.merge(spm, df, left_on=['h_seq', 'pppos'],
                      right_on=['H_SEQ', 'PPPOS'])
        rename = {k: v for k, v in STATA_MAP.items() if k in df.columns}
        df = df.rename(columns=rename)
    
    # Fill missing columns with NaN
    for var in ALL_VARS:
        if var not in df.columns:
            df[var] = np.nan
    
    # Drop records with corrupt/null data (a few rows in some years)
    df = df.dropna(subset=['MARSUPWT', 'A_AGE'])
    
    income_year = asec_year - 1
    print(f'  Read {len(df):,} person records for income year {income_year}')
    return df

In [4]:
def compute_spm_effects(df):
    """Compute anti-poverty effects of each program.
    
    Returns DataFrame with columns: program, total, adults, kids, elderly.
    Values = millions of people removed from (positive) or pushed into
    (negative) poverty by each program.
    """
    age_grps = lambda x: np.where(
        x.A_AGE < 19, 'kids',
        np.where((x.A_AGE > 17) & (x.A_AGE < 65), 'adults', 'elderly'))
    df = df.assign(AGE_GRP=age_grps)
    
    # Combined programs
    df['SPM_RTC'] = df['SPM_ACTC'].fillna(0) + df['SPM_EITC'].fillna(0)
    if 'SPM_EngVal' in df.columns and 'SPM_BBSUBVAL' in df.columns:
        df['SPM_Util'] = df['SPM_EngVal'].fillna(0) + df['SPM_BBSUBVAL'].fillna(0)
    else:
        df['SPM_Util'] = df.get('SPM_EngVal', pd.Series(0, index=df.index)).fillna(0)
    df['SPM_SNAPSL'] = df['SPM_SNAPSub'].fillna(0) + df['SPM_SchLunch'].fillna(0)
    
    # Cash programs: sum by SPM unit
    cash = ['SS', 'SSI', 'UC', 'CSP', 'PAW']
    for i in cash:
        df[f'SPM_{i}'] = df.groupby('SPM_ID')[f'{i}_VAL'].transform('sum')
    
    # Base poverty
    povtot = df.groupby('SPM_Poor').MARSUPWT.sum()[1.0] / 100
    povgrp = df.groupby(['SPM_Poor', 'AGE_GRP']).MARSUPWT.sum()[1.0] / 100
    
    # Programs that reduce poverty (benefit removed = more poverty)
    reductions = cash + ['RTC', 'CapHouseSub', 'WICval', 'SNAPSub',
                         'SchLunch', 'EngVal', 'BBSUBVAL', 'Util', 'SNAPSL']
    
    res = pd.Series(dtype='float')
    res2 = pd.DataFrame()
    
    for i in reductions:
        col = f'SPM_{i}'
        if col not in df.columns or df[col].isna().all():
            continue
        df[f'{col}_Poor'] = ((df['SPM_Resources'] - df[col].fillna(0))
                             < df['SPM_PovThreshold']) * 1
        try:
            povc = df.groupby(f'{col}_Poor').MARSUPWT.sum()[1.0] / 100
            povg = df.groupby([f'{col}_Poor', 'AGE_GRP']).MARSUPWT.sum()[1.0] / 100
        except KeyError:
            continue
        res[i] = (povc - povtot) / 1_000_000
        res2[i] = (povg - povgrp) / 1_000_000
    
    # Programs that add to poverty (cost added = more poverty)
    additions = ['FICA', 'ChildcareXpns', 'MedXpns', 'WkXpns',
                 'FedTaxBC', 'ChildSupPd']
    
    for i in additions:
        col = f'SPM_{i}'
        if col not in df.columns or df[col].isna().all():
            continue
        df[f'{col}_Poor'] = ((df['SPM_Resources'] + df[col].fillna(0))
                             < df['SPM_PovThreshold']) * 1
        try:
            povc = df.groupby(f'{col}_Poor').MARSUPWT.sum()[1.0] / 100
            povg = df.groupby([f'{col}_Poor', 'AGE_GRP']).MARSUPWT.sum()[1.0] / 100
        except KeyError:
            continue
        res[i] = (povc - povtot) / 1_000_000
        res2[i] = (povg - povgrp) / 1_000_000
    
    # Build output table
    tbl = res2.T
    tbl['total'] = res
    
    # Rename programs
    program_names = {
        'SS': 'Social Security', 'SSI': 'SSI',
        'UC': 'Unemployment Insurance',
        'RTC': 'Refundable Tax Credits',
        'SNAPSub': 'SNAP', 'CapHouseSub': 'Housing Subsidies',
        'PAW': 'TANF/General Assistance',
        'SchLunch': 'School Lunch', 'CSP': 'Child Support Received',
        'WICval': 'WIC', 'EngVal': 'Energy Assistance',
        'BBSUBVAL': 'Broadband Subsidy',
        'Util': 'Utilities Assistance',
        'SNAPSL': 'SNAP + School Lunch',
        'FICA': 'FICA', 'FedTaxBC': 'Federal Income Tax',
        'ChildcareXpns': 'Childcare Expenses',
        'MedXpns': 'Medical Expenses',
        'WkXpns': 'Work Expenses',
        'ChildSupPd': 'Child Support Paid',
    }
    tbl = tbl.rename(index=program_names)
    # Negate so positive = removes from poverty
    tbl = -tbl
    tbl.index.name = 'program'
    return tbl.reset_index()

In [5]:
def compute_income_distribution(df):
    """Compute percentile distribution of personal income.
    
    Returns DataFrame with columns: percentile, PTOTVAL, PEARNVAL.
    Values in nominal dollars.
    """
    d = df.query('MARSUPWT > 0').copy()
    
    res = {}
    for val in ['PTOTVAL', 'PEARNVAL']:
        res[val] = {}
        bins = np.arange(-500, 350000, 500)
        cdf = (d.groupby(pd.cut(d[val], bins), observed=False)
                .MARSUPWT.sum().cumsum() / d.MARSUPWT.sum())
        for i in range(0, 99):
            res[val][i] = np.interp(i / 100, cdf, bins[1:])
    
    data = pd.DataFrame(res)
    data.index.name = 'percentile'
    return data.reset_index()

In [6]:
def categorize_population(df):
    """Add Category, poverty indicators, and gap columns to df."""
    df = df.copy()
    
    df['Category'] = np.where(
        df.A_AGE < 18, 'Children',
        np.where(df.A_AGE > 64, 'Elderly',
        np.where(((df.PRDISFLG == 1) | (df.PYRSN == 1) | (df.RSNNOTW == 1)), 'Disabled',
        np.where(((df.PYRSN == 3) | (df.RSNNOTW == 4)), 'Students',
        np.where(((df.PYRSN == 2) | (df.RSNNOTW == 3)), 'Carers',
        np.where(((df.PYRSN == 5) | (df.RSNNOTW == 5) | (df.LKWEEKS > 0)), 'Unemployed',
        np.where(((df.PYRSN == 4) | (df.RSNNOTW == 2)), 'Early Retired',
        np.where(df.WKSWORK > 49, 'Fully Employed', 'All Other'))))))))
    
    df['AGE_GRP'] = pd.cut(df.A_AGE, range(0, 79, 3))
    
    df['SPM'] = np.where(df['SPM_Resources'] < df['SPM_PovThreshold'], 1, 0)
    df['OPM'] = np.where(df['PERLIS'] == 1, 1, 0)
    
    # CSP_VAL excluded: child support received is market income, not a transfer
    market_benefits = [b for b in CASH_BENEFITS if b != 'CSP_VAL']
    benefits_sum = df[market_benefits].fillna(0).sum(axis=1)
    benefits_by_unit = benefits_sum.groupby(df['SPM_ID']).transform('sum')
    df['MARKET_INCOME'] = (
        df['SPM_Totval']
        - df[['SPM_CapWkCCXpns', 'SPM_MedXpns', 'SPM_ChildSupPd']].fillna(0).sum(axis=1)
        - benefits_by_unit
    )
    
    df['SPM_MI'] = np.where(df['MARKET_INCOME'] < df['SPM_PovThreshold'], 1, 0)
    df['MI_GAP'] = ((df['SPM_PovThreshold'] - df['MARKET_INCOME'])
                    / df['SPM_NumPer']) * df['SPM_Weight'] / 100
    df['SPM_GAP'] = ((df['SPM_PovThreshold'] - df['SPM_Resources'])
                     / df['SPM_NumPer']) * df['SPM_Weight'] / 100
    
    return df


def compute_poverty_composition(df):
    """Compute share of poor in each demographic category.
    
    Returns DataFrame: category × (SPM, OPM, SPM_MI) as percent.
    """
    data_spm = df.query('SPM == 1')
    data_opm = df.query('OPM == 1')
    data_mi = df.query('SPM_MI == 1')
    
    results = pd.DataFrame()
    results['SPM'] = (data_spm.groupby('Category').SPM_Weight.sum()
                      / data_spm.SPM_Weight.sum() * 100)
    results['OPM'] = (data_opm.groupby('Category').MARSUPWT.sum()
                      / data_opm.MARSUPWT.sum() * 100)
    results['SPM_MI'] = (data_mi.groupby('Category').SPM_Weight.sum()
                         / data_mi.SPM_Weight.sum() * 100)
    results.index.name = 'category'
    return results.reset_index()


def compute_poverty_rates(df):
    """Compute poverty rate of each demographic category.
    
    Returns DataFrame: category × (SPM, OPM, SPM_MI) as percent.
    """
    results = pd.DataFrame()
    results['SPM'] = (df.groupby('Category', observed=False)
        .apply(lambda x: np.average(x['SPM'], weights=x['SPM_Weight']),
               include_groups=False) * 100)
    results['OPM'] = (df.groupby('Category', observed=False)
        .apply(lambda x: np.average(x['OPM'], weights=x['MARSUPWT']),
               include_groups=False) * 100)
    results['SPM_MI'] = (df.groupby('Category', observed=False)
        .apply(lambda x: np.average(x['SPM_MI'], weights=x['SPM_Weight']),
               include_groups=False) * 100)
    results.index.name = 'category'
    return results.reset_index()

In [7]:
def compute_poverty_gap_by_age(df):
    """Compute poverty gap by 3-year age group.
    
    Returns DataFrame: age × (disposable_gap, market_gap) in billions.
    """
    data_spm = df.query('SPM == 1')
    data_mi = df.query('SPM_MI == 1')
    
    results = pd.DataFrame()
    results['disposable_gap'] = (
        data_spm.groupby('AGE_GRP', observed=False)['SPM_GAP'].sum() / 3e9)
    results['market_gap'] = (
        data_mi.groupby('AGE_GRP', observed=False)['MI_GAP'].sum() / 3e9
        - results['disposable_gap'])
    
    results.index = [i.left for i in results.index]
    results.index.name = 'age'
    return results.reset_index()


def compute_poverty_totals(df):
    """Compute aggregate poverty counts, rates, and gaps.
    
    Returns dict with keys: opm_count, spm_count, mi_count (millions),
    opm_rate, spm_rate, mi_rate (percent),
    spm_gap_total, mi_gap_total (billions).
    """
    opm_count = df.query('OPM == 1').MARSUPWT.sum() / 1e8
    spm_count = df.query('SPM == 1').MARSUPWT.sum() / 1e8
    mi_count = df.query('SPM_MI == 1').MARSUPWT.sum() / 1e8
    
    opm_rate = np.average(df['OPM'], weights=df['MARSUPWT']) * 100
    spm_rate = np.average(df['SPM'], weights=df['MARSUPWT']) * 100
    mi_rate = np.average(df['SPM_MI'], weights=df['MARSUPWT']) * 100
    
    spm_gap = df.query('SPM == 1').SPM_GAP.sum() / 1e9
    mi_gap = df.query('SPM_MI == 1').MI_GAP.sum() / 1e9
    
    return {
        'opm_count': opm_count, 'spm_count': spm_count, 'mi_count': mi_count,
        'opm_rate': opm_rate, 'spm_rate': spm_rate, 'mi_rate': mi_rate,
        'spm_gap_total': spm_gap, 'mi_gap_total': mi_gap,
    }

## Process all years

In [8]:
# Income years to process
income_years = list(range(2014, 2026))  # 2014 through 2025

# Accumulators
all_spm_effects = []
all_income_dist = []
all_poverty_comp = []
all_poverty_rates = []
all_poverty_gap = []
all_poverty_totals = []

for income_year in income_years:
    asec_year = income_year + 1
    if asec_year not in DICT_FILES:
        print(f'Skipping income year {income_year} (no dictionary)')
        continue
    
    print(f'Processing income year {income_year} (ASEC {asec_year})...')
    df = read_asec_person(asec_year)
    
    # 1. SPM effects
    effects = compute_spm_effects(df)
    effects.insert(0, 'year', income_year)
    all_spm_effects.append(effects)
    
    # 2. Income distribution
    dist = compute_income_distribution(df)
    dist.insert(0, 'year', income_year)
    all_income_dist.append(dist)
    
    # 3-6. Poverty demographics and gaps
    df_cat = categorize_population(df)
    
    comp = compute_poverty_composition(df_cat)
    comp.insert(0, 'year', income_year)
    all_poverty_comp.append(comp)
    
    rates = compute_poverty_rates(df_cat)
    rates.insert(0, 'year', income_year)
    all_poverty_rates.append(rates)
    
    gap = compute_poverty_gap_by_age(df_cat)
    gap.insert(0, 'year', income_year)
    all_poverty_gap.append(gap)
    
    totals = compute_poverty_totals(df_cat)
    totals['year'] = income_year
    all_poverty_totals.append(totals)
    
    print(f'  Done: {income_year}')

print(f'\nProcessed {len(all_spm_effects)} years.')

Processing income year 2014 (ASEC 2015)...


  Read 199,024 person records for income year 2014


  Done: 2014
Processing income year 2015 (ASEC 2016)...


  Read 185,487 person records for income year 2015


  Done: 2015
Processing income year 2016 (ASEC 2017)...


...

  Done: 2023
Processing income year 2024 (ASEC 2025)...


  Read 142,125 person records for income year 2024


  Done: 2024
Processing income year 2025 (ASEC 2026)...


  Read 134,729 person records for income year 2025


  Done: 2025

Processed 12 years.


In [9]:
# Combine and write CSVs
spm_effects = pd.concat(all_spm_effects, ignore_index=True)
spm_effects.round(4).to_csv(OUTPUT_PATH / 'asec_spm_effects.csv', index=False)
print(f'asec_spm_effects.csv: {len(spm_effects)} rows')

income_dist = pd.concat(all_income_dist, ignore_index=True)
income_dist.round(2).to_csv(OUTPUT_PATH / 'asec_income_distribution.csv', index=False)
print(f'asec_income_distribution.csv: {len(income_dist)} rows')

poverty_comp = pd.concat(all_poverty_comp, ignore_index=True)
poverty_comp.round(4).to_csv(OUTPUT_PATH / 'asec_poverty_composition.csv', index=False)
print(f'asec_poverty_composition.csv: {len(poverty_comp)} rows')

poverty_rates = pd.concat(all_poverty_rates, ignore_index=True)
poverty_rates.round(4).to_csv(OUTPUT_PATH / 'asec_poverty_rates.csv', index=False)
print(f'asec_poverty_rates.csv: {len(poverty_rates)} rows')

poverty_gap = pd.concat(all_poverty_gap, ignore_index=True)
poverty_gap.round(4).to_csv(OUTPUT_PATH / 'asec_poverty_gap_by_age.csv', index=False)
print(f'asec_poverty_gap_by_age.csv: {len(poverty_gap)} rows')

poverty_totals = pd.DataFrame(all_poverty_totals)
cols = ['year', 'opm_count', 'spm_count', 'mi_count',
        'opm_rate', 'spm_rate', 'mi_rate',
        'spm_gap_total', 'mi_gap_total']
poverty_totals[cols].round(4).to_csv(
    OUTPUT_PATH / 'asec_poverty_totals.csv', index=False)
print(f'asec_poverty_totals.csv: {len(poverty_totals)} rows')

asec_spm_effects.csv: 232 rows
asec_income_distribution.csv: 1188 rows
asec_poverty_composition.csv: 108 rows
asec_poverty_rates.csv: 108 rows
asec_poverty_gap_by_age.csv: 312 rows
asec_poverty_totals.csv: 12 rows


In [10]:
# Preview latest year
latest = income_years[-1]
print(f'=== Income Year {latest} ===')
print(f'\nSPM Effects (top 5):')
yr = spm_effects.query(f'year == {latest}').sort_values('total', ascending=False)
print(yr.head().to_string(index=False))
print(f'\nPoverty Totals:')
print(poverty_totals.query(f'year == {latest}').to_string(index=False))
print(f'\nPoverty Composition:')
print(poverty_comp.query(f'year == {latest}').to_string(index=False))

=== Income Year 2025 ===

SPM Effects (top 5):
 year            program   adults  elderly     kids    total
 2025   Medical Expenses 3.621038 2.476471 1.526710 7.624219
 2025               FICA 2.794910 0.247077 1.434739 4.476725
 2025      Work Expenses 2.268818 0.194700 1.099659 3.563177
 2025 Federal Income Tax 0.982805 0.139668 0.297163 1.419637
 2025 Childcare Expenses 0.237003 0.004559 0.368524 0.610086

Poverty Totals:
 opm_count  spm_count  mi_count  opm_rate  spm_rate   mi_rate  spm_gap_total  mi_gap_total  year
 34.505117  44.390217 79.631782 10.191882 13.111675 23.521084     265.568788    771.488093  2025

Poverty Composition:
 year       category       SPM       OPM    SPM_MI
 2025      All Other  5.608684  6.287940  3.905988
 2025         Carers  7.030142  8.332476  4.594685
 2025       Children 21.193608 27.841848 18.211302
 2025       Disabled 13.490737 15.132689 12.093443
 2025  Early Retired  3.164623  3.072841  3.097382
 2025        Elderly 23.506289 18.469747 40.5855